<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB04_Decision_Trees_Ensembles_and_SVM_with_Real_Sonar_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB04 · Class 4 — Decision Trees, Ensembles, and SVM with Real Sonar Data**

## Block 2: AI — Machine Learning (continued)

`NB03` trained one classifier (Logistic Regression) and one regression pair (Linear Regression vs. Random Forest). This class goes deeper on the classifier side: how decision trees actually split data, how ensembles (bagging and boosting) combine many weak models into a strong one, and how Support Vector Machines find a separating boundary — then compares all of them, properly, on a real dataset: the classic **[Sonar, Mines vs. Rocks](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)** dataset (208 real sonar returns, 60 frequency-band features, genuinely naval/underwater in origin), already mirrored in this repository at [`Datasets/sonar.all-data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/sonar.all-data).

### Learning objectives

By the end of this class, students will be able to:
- Explain how a decision tree splits data (Gini impurity) and why unrestricted trees overfit.
- Distinguish bagging (Random Forest) from boosting (AdaBoost), and explain why ensembles usually beat a single tree.
- Explain the core idea of a Support Vector Machine: maximizing the margin, support vectors, and the kernel trick.
- Build a leakage-safe `Pipeline` that scales data *inside* cross-validation, not before it.
- Fairly compare several classifiers with cross-validation, and read a Random Forest's feature importances.
- Run a first hyperparameter search with `GridSearchCV`.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of NB01–NB03, today's roadmap | 5 min | Theory |
| 2 | Loading and exploring the real dataset (Sonar: Mines vs. Rocks) | 15 min | Practice |
| 3 | Decision trees: how CART works, and a hands-on overfitting demo | 20 min | Theory + Practice |
| 4 | Ensembles: bagging (Random Forest) vs. boosting (AdaBoost) | 15 min | Theory |
| 5 | Support Vector Machines: margin, support vectors, kernels | 10 min | Theory |
| 6 | Preprocessing and leakage-safe pipelines | 10 min | Theory + Practice |
| 7 | Comparing four classifiers with cross-validation | 20 min | Practice |
| 8 | Feature importance from Random Forest | 10 min | Practice |
| 9 | A first hyperparameter search with `GridSearchCV` | 10 min | Theory + Practice |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB01`**: history of AI, why AI matters for naval/ocean engineering.
- **`NB02`**: Python, Colab, NumPy essentials, first Pandas dataset.
- **`NB03`**: the ML workflow end to end — train/val/test, overfitting, metrics, a classifier and a regressor, cross-validation, and a real data-leakage example.
- **`NB04`** (today): specific classification algorithms, in depth, all compared fairly on the same real dataset.

We are still inside **Block 2 — AI: Machine Learning** of the course roadmap.

---

## 2. Loading and exploring the real dataset

The **[Sonar, Mines vs. Rocks](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)** dataset was collected by bouncing sonar signals off a metal cylinder (simulating a mine) and off rocks, at a variety of angles. Each of the 208 records is one sonar return, represented by 60 numbers (energy in 60 frequency bands, each in the 0.0–1.0 range) — a genuinely naval underwater-acoustics problem: telling a mine-like object apart from a rock using sonar alone.

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]
print(sonar.shape)
sonar.head()

`label` is `M` (mine) or `R` (rock). Let's check the class balance and confirm there are no missing values before doing anything else — exactly the same first questions we asked of every dataset in `NB02`/`NB03`.

In [ ]:
print(sonar["label"].value_counts())
print()
print("missing values:", sonar.isna().sum().sum())

We'll build our feature matrix `X` (the 60 frequency bands) and a binary target `y` (1 = mine, 0 = rock) once, and reuse them for every model in this class.

In [ ]:
X = sonar.drop(columns="label")
y = (sonar["label"] == "M").astype(int)

print("X shape:", X.shape, " mine ratio:", y.mean().round(3))

---

## 3. Decision trees

A **decision tree** (CART — Classification and Regression Trees) predicts by asking a sequence of yes/no questions about the features, e.g. "is `freq_11` > 0.18?". At each step, it picks the split that best separates the classes, measured by **Gini impurity**:

$$
\text{Gini} = 1 - \sum_{k} p_k^2
$$

where $p_k$ is the proportion of class $k$ in a node. A pure node (all one class) has Gini = 0; a 50/50 split has the highest impurity. The tree keeps splitting, greedily, to reduce impurity — and, left unchecked, it will keep splitting until every training point is perfectly classified. That is **overfitting**, exactly the concept from `NB03`: a tree that memorizes the training set does poorly on new data.

`max_depth` (and similar parameters like `min_samples_leaf`) control this: shallow trees underfit, unrestricted trees overfit. Let's see it directly, on our real dataset.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

Train decision trees of increasing `max_depth`, and track accuracy on both the training set and the held-out test set:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

depths = range(1, 15)
train_acc, test_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, tree.predict(X_train)))
    test_acc.append(accuracy_score(y_test, tree.predict(X_test)))

plt.plot(depths, train_acc, marker="o", label="Train accuracy")
plt.plot(depths, test_acc, marker="o", label="Test accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision tree: overfitting as depth increases")
plt.legend()
plt.show()

**Read your own plot**: training accuracy should climb toward 1.0 as `max_depth` grows — the tree can always fit its own training data better with more splits. Test accuracy behaves differently: it typically rises at first, then flattens or drops once the tree is deep enough to start memorizing noise. The gap between the two curves *is* overfitting, made visible. Where would you stop growing the tree?

---

## 4. Ensembles: combining many models

A single decision tree is a **high-variance** model: small changes in the training data can produce a very different tree (you likely saw this instability reflected in the jagged test-accuracy curve above). **Ensemble methods** train many models and combine their predictions, which reduces variance without necessarily increasing bias — a direct, practical application of the bias-variance tradeoff from `NB03`.

### Bagging: Random Forest

**Bagging** (Bootstrap Aggregating) trains many trees, each on a random bootstrap sample of the training data, then averages (or votes on) their predictions. **Random Forest** adds one more source of randomness: at each split, it only considers a random subset of features, which decorrelates the trees further — so their errors are less likely to line up, and averaging cancels out more noise.

### Boosting: AdaBoost

**Boosting** builds models *sequentially*: each new model focuses on the training examples the previous ones got wrong (by re-weighting them), then all models vote, weighted by how accurate they were. **AdaBoost** (Adaptive Boosting) is the classic example. Where bagging reduces variance by averaging independent models, boosting reduces bias by chaining models that correct each other's mistakes.

| Method | Models trained | How combined | Mainly reduces | Example |
|---|---|---|---|---|
| Bagging | Independently, in parallel, on bootstrap samples | Average / vote | Variance | Random Forest |
| Boosting | Sequentially, each correcting the last | Weighted vote | Bias | AdaBoost |

Both will appear in our comparison below, alongside the single decision tree, so we can see the effect directly on real accuracy numbers rather than just in theory.

---

## 5. Support Vector Machines

A **Support Vector Machine (SVM)** looks for the boundary between classes that maximizes the **margin** — the distance to the nearest training points of each class. Those nearest points are the **support vectors**; they alone determine where the boundary sits (points far from the boundary don't matter).

Real data is rarely linearly separable, so SVMs commonly use the **kernel trick**: instead of explicitly transforming features into a higher-dimensional space where a straight boundary would work, a kernel function computes the equivalent result directly, without ever forming that expensive transformation. The **RBF (radial basis function) kernel** is the usual default for non-linear problems — it is what we will use below.

SVMs are known for working well on datasets with **many features and relatively few samples** — exactly the shape of our Sonar dataset (208 rows, 60 features) — which is why it's included in this comparison rather than left as a theoretical aside.

---

## 6. Preprocessing and leakage-safe pipelines

SVMs (and distance-based methods generally) are sensitive to feature scale, so we scale the 60 frequency bands with `StandardScaler`, just as we scaled numeric features in `NB03`.

There's a subtlety `NB03` didn't need to deal with: when we cross-validate (train/test many times on different folds), scaling **must be fit only on each fold's training data**, not on the whole dataset beforehand — otherwise information from each fold's test data leaks into the scaler's mean/variance, quietly inflating our accuracy estimate. This is the same *data leakage* idea from `NB03`, applied to a preprocessing step instead of a feature.

`scikit-learn`'s `Pipeline` solves this automatically: bundling `StandardScaler` and a model into one `Pipeline` object makes cross-validation refit the scaler fresh on every fold.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

example_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=42)),
])
example_pipe

---

## 7. Comparing four classifiers with cross-validation

Now let's put everything together: **Decision Tree**, **Random Forest** (bagging), **AdaBoost** (boosting), and **SVM (RBF)** — each wrapped in the same scaling `Pipeline`, evaluated with the same 5-fold stratified cross-validation, on the same real data. This "algorithm spot-checking" approach — trying several reasonable algorithms under identical conditions before committing to one — is standard practice: you rarely know in advance which algorithm will work best on a new dataset.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    cv_results[name] = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

results_df = pd.DataFrame(cv_results)
results_df.mean().sort_values(ascending=False)

The table above shows mean accuracy; it hides how *consistent* each model is across folds. A boxplot shows both at once:

In [ ]:
results_df.boxplot(figsize=(8, 5))
plt.ylabel("Cross-validated accuracy")
plt.title("Algorithm comparison — Sonar: Mines vs. Rocks")
plt.show()

**Interpret your own results**: is the single Decision Tree the weakest of the four, and are Random Forest/AdaBoost both clear improvements over it — consistent with the bias-variance argument in Part 4? Does SVM's box overlap with the ensembles', or is it clearly ahead/behind? A model with a *higher median but wider box* is riskier than one with a *slightly lower median but a tighter box* — which would you trust more for an operational mine-detection system, and why?

---

## 8. Feature importance from Random Forest

Unlike a single opaque prediction, a Random Forest can tell us **which features it relied on most**, averaged across all its trees. Here, that means: which of the 60 frequency bands carry the most information for telling mines apart from rocks.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

Plot the top 15 for a quicker read:

In [ ]:
importances.head(15).plot(kind="barh", figsize=(6, 6))
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top 15 most informative frequency bands (Random Forest)")
plt.show()

If a small handful of frequency bands dominate, that's a hint that a simpler model using only those bands might perform nearly as well — useful if you ever needed to reduce the number of sensor channels or computation on real hardware.

---

## 9. A first hyperparameter search

Every model above used its default settings. `GridSearchCV` automates trying combinations of hyperparameters, using cross-validation to score each one fairly, and keeps the best. For SVM, the two hyperparameters that matter most with an RBF kernel are `C` (how much to penalize misclassified points — smaller means a wider, softer margin) and `gamma` (how far the influence of a single training point reaches — smaller means smoother boundaries).

In [ ]:
from sklearn.model_selection import GridSearchCV

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=42)),
])

param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1],
}

grid = GridSearchCV(svm_pipe, param_grid, cv=cv, scoring="accuracy")
grid.fit(X, y)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

Compare `grid.best_score_` to the plain `SVM (RBF)` result from Part 7 — tuning `C` and `gamma` should match or beat the default settings, since the grid search includes the default (`C=1, gamma="scale"`) as one of its candidates.

---

## Class summary

- Decision trees split on Gini impurity and overfit without a depth limit — we saw the train/test accuracy gap grow directly, on real data.
- Ensembles reduce error two different ways: bagging (Random Forest) reduces variance by averaging independent trees; boosting (AdaBoost) reduces bias by chaining models that fix each other's mistakes.
- SVMs maximize the margin between classes and use kernels to handle non-linear boundaries; they suit datasets with many features and few samples, like ours.
- Scaling must happen *inside* cross-validation, not before it — `Pipeline` makes that automatic and prevents a subtle form of data leakage.
- We fairly compared four classifiers with cross-validation, read a Random Forest's feature importances, and ran a first `GridSearchCV`.

## For the next class (NB05)

We'll switch from supervised to **unsupervised learning**: clustering (K-Means) and dimensionality reduction (PCA) — grouping ships or voyages by similarity without any labels at all.

## Homework / Practice Ideas

1. Add k-Nearest Neighbors and Naive Bayes to the Part 7 comparison — how do they rank against the four models we tried?
2. Repeat the Part 3 overfitting demo, but for `RandomForestClassifier` varying `n_estimators` instead of `DecisionTreeClassifier` varying `max_depth` — does more trees ever start to hurt test accuracy?
3. Extend the Part 9 grid search to include `kernel="linear"` as an option — does a linear kernel do noticeably worse than RBF on this dataset? What would that tell you about the data?
4. Using the feature importances from Part 8, retrain a Random Forest using only the top 10 frequency bands. How much accuracy (if any) do you lose by dropping the other 50 features?
5. In your own words, explain why fitting `StandardScaler` on the full dataset before cross-validation would be a form of data leakage — and what the practical (not just theoretical) consequence would be.

> ***As always: keep every question framed around what an operational sonar/mine-detection system would actually need — not just "which number is highest".***
